# Data Leakage Audit

 **Objective**: Ensure that the machine learning models are not cheating by exploiting data leakage.
 This includes checking for overlapping captures between splits, exact duplicate flows across
 train/test, and ensuring no prohibited metadata (like IPs, ports, or timestamps) has leaked
 into the feature space.

In [1]:
%matplotlib inline

import pandas as pd
import numpy as np

from src.pipeline.data_preparation import load_and_prepare_data
from src.pipeline.feature_pipeline import FeaturePipeline
from src.utils.logging import setup_logger

logger = setup_logger(level="INFO")

In [2]:
logger.info("Loading datasets...")
df_all = load_and_prepare_data()

df_train = df_all[df_all["split"] == "train"].copy()
df_val = df_all[df_all["split"] == "val"].copy()
df_test = df_all[df_all["split"] == "test"].copy()

2026-03-30 12:14:14 | INFO | ai-vpn-firewall | Loading datasets...
2026-03-30 12:14:14 | INFO | ai-vpn-firewall | Loading VNAT (PCAP-based)...
2026-03-30 12:14:14 | INFO | ai-vpn-firewall | [VNAT] Loaded features.parquet (33711 flows, splits already assigned)
2026-03-30 12:14:14 | INFO | ai-vpn-firewall | Loading ISCX (PCAP-based)...
2026-03-30 12:14:14 | INFO | ai-vpn-firewall | [ISCX] Loaded features.parquet (76687 flows, splits already assigned)
2026-03-30 12:14:14 | INFO | ai-vpn-firewall | Loading USBVPN (JSON-based)...
2026-03-30 12:14:14 | INFO | ai-vpn-firewall | Removing exact duplicate flows across feature columns...
2026-03-30 12:14:14 | INFO | ai-vpn-firewall | Ensuring numeric dtypes for feature columns...
2026-03-30 12:14:14 | INFO | ai-vpn-firewall | ✓ All feature columns successfully converted to numeric dtypes
2026-03-30 12:14:14 | INFO | ai-vpn-firewall | Removed 5750 duplicate flows (7.92%)
2026-03-30 12:14:14 | INFO | ai-vpn-firewall | Metadata columns present for a

## 1. Group Leakage: Capture ID Overlap

In network traffic classification, splitting by `capture_id` (or PCAP file) is critical.
Random splitting allows packets from the same session or background noise to exist in both
Train and Test, causing massive leakage.

In [3]:
logger.info("Running Group Leakage Check...")

train_caps = set(df_train["capture_id"].unique())
val_caps = set(df_val["capture_id"].unique())
test_caps = set(df_test["capture_id"].unique())

2026-03-30 12:14:15 | INFO | ai-vpn-firewall | Running Group Leakage Check...


In [4]:
train_test_overlap = train_caps.intersection(test_caps)
train_val_overlap = train_caps.intersection(val_caps)
val_test_overlap = val_caps.intersection(test_caps)

In [5]:
print("--- Capture ID Overlap Results ---")
print(f"Train / Test Overlap: {len(train_test_overlap)} captures")
if train_test_overlap:
    print(f"  -> Leakage Detected! Overlapping captures: {train_test_overlap}")

print(f"Train / Val Overlap:  {len(train_val_overlap)} captures")
if train_val_overlap:
    print(f"  -> Leakage Detected! Overlapping captures: {train_val_overlap}")

print(f"Val / Test Overlap:   {len(val_test_overlap)} captures")
if val_test_overlap:
    print(f"  -> Leakage Detected! Overlapping captures: {val_test_overlap}")

if not train_test_overlap and not train_val_overlap and not val_test_overlap:
    print("\nSUCCESS: No capture groups leak across splits. The dataset is strictly grouped.")

--- Capture ID Overlap Results ---
Train / Test Overlap: 0 captures
Train / Val Overlap:  0 captures
Val / Test Overlap:   0 captures

SUCCESS: No capture groups leak across splits. The dataset is strictly grouped.


## 2. Feature Schema Metadata Leakage

Ensure that no metadata features (ports, IPs, connection strings, timestamps)
accidentally made it into the `FeaturePipeline` model input.

In [6]:
logger.info("Running Feature Schema Metadata Leakage Check...")

pipe = FeaturePipeline().fit(df_train)
model_features = pipe.model_feature_names()

prohibited_keywords = [
    "ip", "port", "mac", "time", "stamp",
    "id", "capture", "file", "split", "dataset", "source_capture"
]

leaked_features = []
for f in model_features:
    for kw in prohibited_keywords:
        if kw in f.lower():
            leaked_features.append(f)
            break

print("--- Metadata Leakage Results ---")
if leaked_features:
    raise ValueError(f"Metadata leakage detected in model feature space: {leaked_features}")
else:
    print("SUCCESS: No metadata keywords found in the model feature space.")

2026-03-30 12:14:15 | INFO | ai-vpn-firewall | Running Feature Schema Metadata Leakage Check...
--- Metadata Leakage Results ---
SUCCESS: No metadata keywords found in the model feature space.


## 3. Exact Duplicate Flows Across Splits

If identical flow behaviors are in both Train and Test, it inflates test performance.

In [7]:
logger.info("Running Duplicate Flow Check...")

# We check duplicates based only on the core behavioral features
# (ignoring IDs and labels)
X_train_feats = pipe.transform(df_train)[model_features]
X_test_feats = pipe.transform(df_test)[model_features]

2026-03-30 12:14:15 | INFO | ai-vpn-firewall | Running Duplicate Flow Check...


In [8]:
# Merge to find exact matches
merged = X_train_feats.merge(X_test_feats, how="inner", indicator=False)
duplicates_count = len(merged)

In [9]:
print("--- Exact Duplicate Flows Check ---")
print(f"Identical flows existing in both Train and Test: {duplicates_count}")

--- Exact Duplicate Flows Check ---
Identical flows existing in both Train and Test: 0


In [10]:
# Calculate percentage of test set that is leaked
if len(X_test_feats) > 0:
    leak_pct = (duplicates_count / len(X_test_feats)) * 100
    print(f"Percentage of Test Set leaking from Train: {leak_pct:.2f}%")

    if leak_pct > 5.0:
        print("WARNING: High duplicate flow rate. Model might be memorizing.")
    else:
        print("SUCCESS: Duplicate flow rate is within acceptable bounds for network traffic.")

Percentage of Test Set leaking from Train: 0.00%
SUCCESS: Duplicate flow rate is within acceptable bounds for network traffic.


## 4. High Target Correlation (Target Leakage)

Check if any single feature is almost perfectly correlated with the target label.
If a single feature has an absolute correlation > 0.90, it might be a spurious artifact
rather than a true behavioral indicator.

In [11]:
logger.info("Running Target Correlation Check...")

train_corr_df = X_train_feats.copy()
train_corr_df["LABEL_TARGET"] = df_train["label"].astype(int).values

2026-03-30 12:14:15 | INFO | ai-vpn-firewall | Running Target Correlation Check...


In [12]:
correlations = (
    train_corr_df.corr()["LABEL_TARGET"]
    .drop("LABEL_TARGET")
    .abs()
    .sort_values(ascending=False)
)

In [13]:
print("--- Top 5 Features Correlated with Target ---")
print(correlations.head(5))

--- Top 5 Features Correlated with Target ---
dispersion_symmetry          0.303310
sz_p75_median_ratio          0.112893
sz_iqr_norm_median           0.106976
sz_coef_variation            0.066088
direction_balance_packets    0.051959
Name: LABEL_TARGET, dtype: float64


In [14]:
highly_correlated = correlations[correlations > 0.90]

if not highly_correlated.empty:
    print(
        f"\nWARNING: Found {len(highly_correlated)} features with >0.90 correlation to the label. Investigate for target leakage.")
    print(highly_correlated)
else:
    print("\nSUCCESS: No single feature perfectly predicts the target (max correlation < 0.90).")


SUCCESS: No single feature perfectly predicts the target (max correlation < 0.90).
